In [37]:
import pandas as pd
import numpy as np

## Output CSV ##
outfile = "../data/processed/haunted_places_features_added.tab"

# Reading Haunted Places Dataset
haunted_places_df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep="\t")

# Get the 2 datasets (Mental Health and Haunted Places)
mental_health_path = "../data/joined_datasets/mental_health_data.csv"

# Load datasets
mental_health_df = pd.read_csv(mental_health_path)

# Feature Names from Mental Health Dataset
feature_names = ["MENTHLTH", "POORHLTH", "ADDEPEV3", "SDHSTRE1"]

# Print out first few rows of the datasets to verify correct loading
print("Mental Health Dataset:")
print(mental_health_df.head())

print("\nHaunted Places Dataset:")
print(haunted_places_df.head()) 

# State Name to FIPS Code Mapping (Example mapping, please update this if you need a full list)
state_name_to_fips = {
    "Alabama": 1, "Alaska": 2, "Arizona": 4, "Arkansas": 5, "California": 6,
    "Colorado": 8, "Connecticut": 9, "Delaware": 10, "District of Columbia": 11,
    "Florida": 12, "Georgia": 13, "Hawaii": 15, "Idaho": 16, "Illinois": 17,
    "Indiana": 18, "Iowa": 19, "Kansas": 20, "Louisiana": 22, "Maine": 23,
    "Maryland": 24, "Massachusetts": 25, "Michigan": 26, "Minnesota": 27,
    "Mississippi": 28, "Missouri": 29, "Montana": 30, "Nebraska": 31,
    "Nevada": 32, "New Hampshire": 33, "New Jersey": 34, "New Mexico": 35,
    "New York": 36, "North Carolina": 37, "North Dakota": 38, "Ohio": 39,
    "Oklahoma": 40, "Oregon": 41, "Rhode Island": 44, "South Carolina": 45,
    "South Dakota": 46, "Tennessee": 47, "Texas": 48, "Utah": 49, "Vermont": 50,
    "Virginia": 51, "Washington": 53, "West Virginia": 54, "Wisconsin": 55,
    "Wyoming": 56, "Guam": 66, "Puerto Rico": 72, "Virgin Islands": 78
}

# Convert 'state' column in Haunted Places dataset from name to FIPS code
haunted_places_df['state_fips'] = haunted_places_df['state'].map(state_name_to_fips)

# Verify if the 'state_fips' column was created correctly
print("\nHaunted Places Dataset (after mapping state to FIPS):")
print(haunted_places_df[['state', 'state_fips']].head())

# Standardize column names for merging
# Renaming 'state' column in the mental_health_df to avoid conflict
mental_health_df.rename(columns={"_STATE": "mental_health_state"}, inplace=True)

# Verify the renaming was successful
print("\nMental Health Dataset (after renaming 'state' column):")
print(mental_health_df.head())

# Ensure the 'state_fips' column in Haunted Places is string, and 'mental_health_state' is string in Mental Health
haunted_places_df['state_fips'] = haunted_places_df['state_fips'].astype(str)
mental_health_df['mental_health_state'] = mental_health_df['mental_health_state'].astype(str)

# Convert MENTHLTH and POORHLTH to numeric, handling specific values
def convert_mental_health(val):
    if val == 88:  # none
        return 0
    elif val in [77, 99]:  # Don't know, Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

def convert_poor_health(val):
    if val == 88:  # none
        return 0
    elif val in [77, 99]:  # Don't know, Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

def convert_depressive_disorder(val):
    if val == 1:  # Yes
        return 1
    elif val == 2:  # No
        return 0
    elif val in [7, 9]:  # Don't know, Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

def convert_stress(val):
    if val == 7:  # Don't know
        return np.nan
    elif val == 9:  # Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

# Apply conversions
mental_health_df['MENTHLTH'] = mental_health_df['MENTHLTH'].apply(convert_mental_health)
mental_health_df['POORHLTH'] = mental_health_df['POORHLTH'].apply(convert_poor_health)
mental_health_df['ADDEPEV3'] = mental_health_df['ADDEPEV3'].apply(convert_depressive_disorder)
mental_health_df['SDHSTRE1'] = mental_health_df['SDHSTRE1'].apply(convert_stress)

# Merge the datasets on the FIPS code column
merged_df = pd.merge(haunted_places_df, mental_health_df[['mental_health_state', 'MENTHLTH', 'POORHLTH', 'ADDEPEV3', 'SDHSTRE1']], left_on="state_fips", right_on="mental_health_state", how="inner")

# Calculate averages or other metrics by state (e.g., mean for SDHSTRE1 and ADDEPEV3, sum for MENTHLTH and POORHLTH)
state_avg_df = merged_df.groupby('state_fips')[['MENTHLTH', 'POORHLTH', 'ADDEPEV3', 'SDHSTRE1']].mean().reset_index()

# Save the averaged dataset for verification
state_avg_df.to_csv("../data/processed/state_average_mental_health.csv", index=False)

# Read Feature-added dataframe
out_df = pd.read_csv(f"{outfile}", sep="\t")

# Check if features exist and update them in the out_df
for feature in feature_names:
    if feature in out_df.columns:
        # Update values from state_avg_df
        out_df[feature].update(state_avg_df[feature].values)
    else:
        # If the feature doesn't exist, add it from state_avg_df
        out_df[feature] = state_avg_df[feature]

# Save the updated dataset with the new features
out_df.to_csv(f"{outfile}", sep="\t", index=False)

print(f"CSV Saved to {outfile}")

# Display the averaged dataframe to verify
print(state_avg_df[['state_fips', 'MENTHLTH', 'POORHLTH', 'ADDEPEV3', 'SDHSTRE1']].head())

# Preview the updated dataframe with the new features
print("\nUpdated DataFrame with New Features:")
print(out_df.head())

Mental Health Dataset:
   _STATE  FMONTH        IDATE IMONTH   IDAY    IYEAR  DISPCODE  \
0     1.0     1.0  b'03012023'  b'03'  b'01'  b'2023'    1100.0   
1     1.0     1.0  b'01062023'  b'01'  b'06'  b'2023'    1100.0   
2     1.0     1.0  b'03082023'  b'03'  b'08'  b'2023'    1100.0   
3     1.0     1.0  b'03062023'  b'03'  b'06'  b'2023'    1100.0   
4     1.0     1.0  b'01062023'  b'01'  b'06'  b'2023'    1100.0   

           SEQNO          _PSU  CTELENM1  ...  _RFBING6      _DRNKWK2  \
0  b'2023000001'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
1  b'2023000002'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
2  b'2023000003'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
3  b'2023000004'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
4  b'2023000005'  2.023000e+09       1.0  ...       1.0  4.700000e+01   

   _RFDRHV8  _FLSHOT7  _PNEUMO3  _AIDTST4  _RFSEAT2  _RFSEAT3  _DRNKDRV  state  
0       1.0       2.0       2.0       2.0       1.0   


KeyboardInterrupt



In [21]:
import pandas as pd
import numpy as np

## Output CSV ##
outfile = "../data/processed/haunted_places_features_added.tab"

# Reading Haunted Places Dataset
haunted_places_df = pd.read_csv("../data/processed/haunted_places_cleaned.tab", sep="\t")

# Get the 2 datasets (Mental Health and Haunted Places)
mental_health_path = "../data/joined_datasets/mental_health_data.csv"

# Load datasets
mental_health_df = pd.read_csv(mental_health_path)

# Feature Names from Mental Health Dataset
feature_names = ["MENTHLTH", "POORHLTH", "ADDEPEV3", "SDHSTRE1"]

# Print out first few rows of the datasets to verify correct loading
print("Mental Health Dataset:")
print(mental_health_df.head())

print("\nHaunted Places Dataset:")
print(haunted_places_df.head())

Mental Health Dataset:
   _STATE  FMONTH        IDATE IMONTH   IDAY    IYEAR  DISPCODE  \
0     1.0     1.0  b'03012023'  b'03'  b'01'  b'2023'    1100.0   
1     1.0     1.0  b'01062023'  b'01'  b'06'  b'2023'    1100.0   
2     1.0     1.0  b'03082023'  b'03'  b'08'  b'2023'    1100.0   
3     1.0     1.0  b'03062023'  b'03'  b'06'  b'2023'    1100.0   
4     1.0     1.0  b'01062023'  b'01'  b'06'  b'2023'    1100.0   

           SEQNO          _PSU  CTELENM1  ...  _RFBING6      _DRNKWK2  \
0  b'2023000001'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
1  b'2023000002'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
2  b'2023000003'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
3  b'2023000004'  2.023000e+09       1.0  ...       1.0  5.397605e-79   
4  b'2023000005'  2.023000e+09       1.0  ...       1.0  4.700000e+01   

   _RFDRHV8  _FLSHOT7  _PNEUMO3  _AIDTST4  _RFSEAT2  _RFSEAT3  _DRNKDRV  state  
0       1.0       2.0       2.0       2.0       1.0   

In [27]:
# State Name to FIPS Code Mapping (Example mapping, please update this if you need a full list)
state_name_to_fips = {
    "Alabama": 1, "Alaska": 2, "Arizona": 4, "Arkansas": 5, "California": 6,
    "Colorado": 8, "Connecticut": 9, "Delaware": 10, "District of Columbia": 11,
    "Florida": 12, "Georgia": 13, "Hawaii": 15, "Idaho": 16, "Illinois": 17,
    "Indiana": 18, "Iowa": 19, "Kansas": 20, "Louisiana": 22, "Maine": 23,
    "Maryland": 24, "Massachusetts": 25, "Michigan": 26, "Minnesota": 27,
    "Mississippi": 28, "Missouri": 29, "Montana": 30, "Nebraska": 31,
    "Nevada": 32, "New Hampshire": 33, "New Jersey": 34, "New Mexico": 35,
    "New York": 36, "North Carolina": 37, "North Dakota": 38, "Ohio": 39,
    "Oklahoma": 40, "Oregon": 41, "Rhode Island": 44, "South Carolina": 45,
    "South Dakota": 46, "Tennessee": 47, "Texas": 48, "Utah": 49, "Vermont": 50,
    "Virginia": 51, "Washington": 53, "West Virginia": 54, "Wisconsin": 55,
    "Wyoming": 56, "Guam": 66, "Puerto Rico": 72, "Virgin Islands": 78
}

In [35]:
# Convert 'state' column in Haunted Places dataset from name to FIPS code
haunted_places_df['state_fips'] = haunted_places_df['state'].map(state_name_to_fips)

# Standardize column names for merging
mental_health_df.rename(columns={"_STATE": "mental_health_state"}, inplace=True)

# Ensure the 'state_fips' column in Haunted Places is string, and 'mental_health_state' is string in Mental Health
haunted_places_df['state_fips'] = haunted_places_df['state_fips'].astype(str)
mental_health_df['mental_health_state'] = mental_health_df['mental_health_state'].astype(str)

KeyError: 'mental_health_state'

In [31]:
# Convert MENTHLTH and POORHLTH to numeric, handling specific values
def convert_mental_health(val):
    if val == 88:  # none
        return 0
    elif val in [77, 99]:  # Don't know, Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

def convert_poor_health(val):
    if val == 88:  # none
        return 0
    elif val in [77, 99]:  # Don't know, Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

def convert_depressive_disorder(val):
    if val == 1:  # Yes
        return 1
    elif val == 2:  # No
        return 0
    elif val in [7, 9]:  # Don't know, Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

def convert_stress(val):
    if val == 7:  # Don't know
        return np.nan
    elif val == 9:  # Refused
        return np.nan
    elif isinstance(val, str) and val.strip() == '':  # Blank
        return np.nan
    else:
        return val

# Apply conversions
mental_health_df['MENTHLTH'] = mental_health_df['MENTHLTH'].apply(convert_mental_health)
mental_health_df['POORHLTH'] = mental_health_df['POORHLTH'].apply(convert_poor_health)
mental_health_df['ADDEPEV3'] = mental_health_df['ADDEPEV3'].apply(convert_depressive_disorder)
mental_health_df['SDHSTRE1'] = mental_health_df['SDHSTRE1'].apply(convert_stress)

In [33]:
# Merge the datasets on the 'state_fips' column
merged_df = pd.merge(haunted_places_df, mental_health_df[['state', 'MENTHLTH', 'POORHLTH', 'ADDEPEV3', 'SDHSTRE1']], left_on="state_fips", right_on="state", how="inner")

# Calculate averages or other metrics by state (e.g., mean for SDHSTRE1 and ADDEPEV3, sum for MENTHLTH and POORHLTH)
state_avg_df = merged_df.groupby('state')[['MENTHLTH', 'POORHLTH', 'ADDEPEV3', 'SDHSTRE1']].mean().reset_index()

# Save the averaged dataset for verification
state_avg_df.to_csv("../data/processed/state_average_mental_health.csv", index=False)

# Read Feature-added dataframe
out_df = pd.read_csv(f"{outfile}", sep="\t")

# Check if features exist and update them in the out_df
for feature in feature_names:
    if feature in out_df.columns:
        # Update values from state_avg_df
        out_df[feature].update(state_avg_df[feature].values)
    else:
        # If the feature doesn't exist, add it from state_avg_df
        out_df[feature] = state_avg_df[feature]

# Save the updated dataset with the new features
out_df.to_csv(f"{outfile}", sep="\t", index=False)

print(f"CSV Saved to {outfile}")

# Display the averaged dataframe to verify
print(state_avg_df[['state', 'MENTHLTH', 'POORHLTH', 'ADDEPEV3', 'SDHSTRE1']].head())

# Preview the updated dataframe with the new features
print("\nUpdated DataFrame with New Features:")
print(out_df.head())

ValueError: The column label 'state' is not unique.